<a href="https://colab.research.google.com/github/acastellanos-ie/NLP-MBDS-EN-NEW/blob/main/06_transfer_learning_and_fine_tuning/fine_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transfer Learning and Fine-Tuning

This is the session where Helcon's support copilot gets its triage classifier. Every hour a fresh wave of tickets lands in the shared inbox (a duplicate charge here, a dead webhook there, a password nobody can reset) and someone has to read each one and drop it into the right queue. Here we build the model that does that routing automatically, and we build it five different ways so the build-vs-buy decision comes off a table instead of off opinions. The winner is not a toy: its adapter is the exact triage tool that session 08's agent will call and session 09's capstone will assemble into the Helcon Support Copilot.

## Setup

Run the next cell first and let it work while we talk. It clones the Helcon repository (so the ticket corpus and the shared SDK (system_io) are on disk), installs the libraries, and pre-downloads the three models of the session, so every later cell loads from local cache instead of from the network. If you are running locally with the repository already checked out, the clone is skipped and only the model downloads happen.

In [ ]:
REPO = "https://github.com/acastellanos-ie/NLP-MBDS-EN-NEW"
import os, sys
if not os.path.exists("NLP-MBDS-EN"):
    !git clone --depth 1 -q $REPO NLP-MBDS-EN
sys.path.insert(0, "NLP-MBDS-EN/system")

!pip install -q transformers datasets evaluate accelerate peft scikit-learn
!pip uninstall -q -y torchao torchvision  # avoid peft's torchao>=0.16 gate vs Colab's preinstalled torchao 0.10

# Repair a possibly-mismatched NumPy left by earlier installs (the "_center from
# numpy._core.umath" error). No-op on a clean runtime; otherwise fix it and restart once.
import importlib
try:
    importlib.import_module("numpy._core.strings")
except Exception:
    import os
    get_ipython().system('pip install -qqq --force-reinstall numpy')
    print('>> NumPy was inconsistent - repaired it. The runtime will restart now;')
    print('>> just run THIS setup cell again when it reconnects.')
    os.kill(os.getpid(), 9)

from huggingface_hub import snapshot_download
for r in ["roberta-base", "facebook/bart-large-mnli", "Qwen/Qwen2.5-1.5B-Instruct"]:
    snapshot_download(r)
    print(f"cached: {r}")

from system_io import load_seed, save_artifact, system_status
system_status()

## Step 1: The tickets

Everything starts with the corpus the whole course shares: 748 real-shaped Helcon support tickets, each already labelled with one of ten categories. We load them through the shared SDK (system_io) so this notebook reads exactly the same data as every other session.

In [ ]:
import pandas as pd

tickets = pd.DataFrame(load_seed("tickets"))
taxonomy = load_seed("taxonomy")
CATEGORIES = list(taxonomy["categories"].keys())

print(f"{len(tickets)} tickets, {len(CATEGORIES)} categories")
tickets.head()

Two things to look at before modelling: how many tickets each category has, and how long a ticket is. The triage queues are not balanced (billing complaints arrive far more often than outage reports), and that imbalance is exactly why we will not judge anything on plain accuracy later.

In [ ]:
import matplotlib.pyplot as plt

counts = tickets["category"].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
counts.sort_values().plot.barh(ax=axes[0], color="steelblue")
axes[0].set_title(f"Tickets per category (n = {len(tickets)})")
axes[0].set_xlabel("count")

ticket_words = tickets["text"].str.split().str.len()
axes[1].hist(ticket_words, bins=range(0, ticket_words.max() + 2), color="salmon", edgecolor="white")
axes[1].set_title(f"Ticket length (median {int(ticket_words.median())} words)")
axes[1].set_xlabel("words per ticket")
plt.tight_layout()
plt.show()

print(f"most common : {counts.idxmax()} ({counts.max()})   rarest: {counts.idxmin()} ({counts.min()})")

A glance at three raw tickets. Short, polite, customer-shaped text, with the gold category beside each.

In [ ]:
for _, row in tickets.sample(3, random_state=42).iterrows():
    print(f"[{row['category']:<14}] {row['text']}")

Now the uncomfortable part. The label set assumes every ticket belongs to exactly one queue, but real tickets do not respect that. The seed corpus ships a set of curated hard cases, and the multi-category ones each genuinely span two queues. Single-label triage will be forced to pick one of them, and that built-in lossiness is a ceiling no model in this session can climb past.

In [ ]:
probes = load_seed("probes")

print("Taxonomy (the ten queues):")
for name, info in taxonomy["categories"].items():
    print(f"  {name:<16} {info['description']}")

# The hard cases: each ticket below honestly belongs to TWO queues at once.
print("\nMulti-category hard cases (each one straddles two queues):")
for p in probes["multi_category"]:
    print(f"  [{p['id']}] {p['text']}")
    print(f"           → {p['shows']}")

## Step 2: The arena and the leaderboard

A tournament is only honest if every contender is judged on the same data with the same metric, so we fix both before building anything. One canonical seed-42 stratified train/validation split that **everything** downstream consumes (the sklearn model and the Transformers alike), and the whole validation split as the fixed **arena**. Stratifying keeps the rare categories present on both sides.

The metric is **macro-F1**: the F1 of each category averaged with equal weight. On an imbalanced problem like this it is the fair choice. A model that nails BILLING and ignores OUTAGE cannot hide behind a high accuracy. Place your bet now on the final ranking; we will check it at the end.

In [ ]:
from sklearn.model_selection import train_test_split

SEED = 42

train_df, val_df = train_test_split(
    tickets, test_size=0.2, random_state=SEED, stratify=tickets["category"]
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

# The arena: the WHOLE validation split. Every approach this session is judged on it.
arena_df = val_df.copy()

print(f"train: {len(train_df)} tickets | arena (validation): {len(arena_df)} tickets")
print(f"categories present in the arena: {arena_df['category'].nunique()} / {len(CATEGORIES)}")

In [ ]:
import pandas as pd

RESULTS = {}

def post_score(name, macro_f1, train_seconds=None, ms_per_ticket=None):
    RESULTS[name] = {"macro_f1": macro_f1, "train_seconds": train_seconds, "ms_per_ticket": ms_per_ticket}
    print(f"posted to leaderboard: {name} (macro-F1 = {macro_f1:.3f})")

def show_leaderboard():
    print(f"{'approach':<40} {'macro-F1':>9} {'train (s)':>10} {'ms/ticket':>10}")
    print("-" * 71)
    for name, r in sorted(RESULTS.items(), key=lambda kv: kv[1]["macro_f1"], reverse=True):
        ts = "-" if r["train_seconds"] is None else f"{r['train_seconds']:.0f}"
        ms = "-" if r["ms_per_ticket"] is None else f"{r['ms_per_ticket']:.1f}"
        print(f"{name:<40} {r['macro_f1']:>9.3f} {ts:>10} {ms:>10}")

## Step 3: The classic baseline

Before any Transformer, establish the performance ceiling of the pre-deep-learning era: TF-IDF features feeding a multinomial Logistic Regression. It treats each ticket as a bag of words, ignores order entirely, trains in seconds on a CPU, and predicts in a fraction of a millisecond. That cheapness is the whole point. It is the bar everything heavier has to clear to justify its cost.

The one knob is whether we stem the words first. Stemming collapses *cancelling*, *cancelled* and *cancel* to one token; conventional wisdom says that helps. Flip it and see.

In [ ]:
USE_STEMMING = True   # try: False  (does the hand-crafted cleanup actually move the score?)

import re
from nltk.stem import PorterStemmer

_stemmer = PorterStemmer()

# Cheap, hand-written text cleanup: keep letters only, lowercase, optionally stem.
def preprocess(text):
    words = re.sub(r"[^a-zA-Z]", " ", text).lower().split()
    if USE_STEMMING:
        words = [_stemmer.stem(w) for w in words]   # cancelling/cancelled/cancel -> one token
    return " ".join(words)

In [ ]:
import time
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score

clf = Pipeline([
    ("tfidf", TfidfVectorizer(preprocessor=preprocess, ngram_range=(1, 2), min_df=2)),
    ("lr", LogisticRegression(max_iter=1000, C=10.0)),
])

t0 = time.time()
clf.fit(train_df["text"], train_df["category"])
tfidf_train_seconds = time.time() - t0

t0 = time.time()
tfidf_preds = clf.predict(arena_df["text"])
tfidf_ms = (time.time() - t0) / len(arena_df) * 1000

tfidf_f1 = f1_score(arena_df["category"], tfidf_preds, average="macro")
print(f"TF-IDF + LogisticRegression | arena macro-F1 = {tfidf_f1:.4f}")
post_score("TF-IDF + LogReg", tfidf_f1, tfidf_train_seconds, tfidf_ms)

## Takeaway

The classic stack trains in seconds on a CPU, predicts in a fraction of a millisecond per ticket, and posts a perfectly respectable macro-F1. It is a legitimate production option, not a straw man. Flip `USE_STEMMING` and re-run: the preprocessing we crafted by hand moves the score less than most people expect, the quiet lesson of the bag-of-words era. What it cannot do is read word order or context, and that is exactly where the next contenders earn their cost.

## Step 4: Humans versus the machine

Before we reach for a Transformer, the room plays the classifier. The next cell prints a sample of arena tickets with no labels, including the multi-category probes that genuinely span two queues. Read each one and write down the single category you would route it to. Keep your answers; the reveal compares the room against the gold labels and against the TF-IDF model.

In [ ]:
JURY_SEED = 7   # seed-fixed so the whole room argues over the same tickets

# Pull the multi-category probes that live in the arena, plus a handful of ordinary tickets.
probe_ids = [p["id"] for p in probes["multi_category"]]
jury_probes = arena_df[arena_df["message_id"].isin(probe_ids)]
jury_rest = arena_df.drop(jury_probes.index).sample(n=6, random_state=JURY_SEED)
jury_df = pd.concat([jury_probes, jury_rest]).sample(frac=1, random_state=JURY_SEED).reset_index(drop=True)

print("Route each ticket to ONE category and keep your answer:")
print(f"options: {', '.join(CATEGORIES)}\n")
for i, text in enumerate(jury_df["text"], 1):
    print(f"{i:>2}. {text}")

In [ ]:
jury_model_preds = clf.predict(jury_df["text"])

print(f"{'#':>2}  {'gold':<16} {'TF-IDF model':<16} ticket")
print("-" * 90)
for i, (row, pred) in enumerate(zip(jury_df.itertuples(), jury_model_preds), 1):
    flag = "ok  " if pred == row.category else "MISS"
    print(f"{i:>2}. [{flag}] {row.category:<14} {pred:<16} {row.text[:60]}")

The disagreements are the lesson. On the everyday tickets the room and the model mostly agree. On the multi-category probes ("I was charged twice and now I cannot log in") even the humans split, because the ticket is honestly BILLING **and** ACCOUNT. The taxonomy forces one label, so the gold answer is itself a judgement call. That ceiling is baked into the problem: no model here can be "right" on a ticket whose truth is two categories, and you will see every contender trip on exactly these.

## Step 5: Fine-tuning RoBERTa

Now the heavy contender. RoBERTa already speaks English from its pre-training; fine-tuning nudges it to route Helcon tickets. First we wrap the canonical split in Hugging Face datasets and tokenize. The labels are strings, so we build the `id2label` / `label2id` maps the model needs, and those same maps become a shipped artifact later, so the agent in session 08 can turn a predicted id back into "BILLING".

In [ ]:
from datasets import Dataset, DatasetDict

id2label = {i: c for i, c in enumerate(CATEGORIES)}
label2id = {c: i for i, c in enumerate(CATEGORIES)}

def to_hf(df):
    d = df[["text", "category"]].copy()
    d["label"] = d["category"].map(label2id)
    return Dataset.from_pandas(d[["text", "label"]], preserve_index=False)

dataset = DatasetDict({"train": to_hf(train_df), "validation": to_hf(arena_df)})
print(dataset)

In [ ]:
from transformers import AutoTokenizer, DataCollatorWithPadding

checkpoint = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)   # instant: the setup cell cached it
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)   # pads each batch to its longest item

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True)

tokenized = dataset.map(tokenize, batched=True)

### The learning-rate catastrophe

The learning rate controls how big a step we take on each error: too low and learning crawls, too high and the model over-corrects until it destroys what it already knew. For fine-tuning the rates are tiny ($2\times10^{-5}$) because the model already speaks English. We are nudging it, not rebuilding it.

A claim like "too high destroys the model" deserves a measurement, so before the real run we fire a 100-step probe. Leave the knob at `2e-5` and the loss slides down. Then set it to `5e-3` (250 times the textbook value) and watch the loss column do something far more interesting than exploding.

In [ ]:
LEARNING_RATE = 2e-5   # try: 5e-3  (250x the textbook value; watch the loss column)

from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

probe_model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint, num_labels=len(CATEGORIES), id2label=id2label, label2id=label2id
)

# A short probe run, not the real fine-tune: 100 steps is enough to see the loss behave.
probe_args = TrainingArguments(
    output_dir="./lr_probe",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=16,
    max_steps=100,                 # a short probe, not a full run
    weight_decay=0.01,
    logging_steps=10,
    report_to="none",
    fp16=True,
    seed=SEED,
)

probe_trainer = Trainer(
    model=probe_model,
    args=probe_args,
    train_dataset=tokenized["train"],
    data_collator=data_collator,
)
probe_trainer.train()

At `2e-5` the loss starts high and slides steadily down within the hundred steps. At `5e-3` it does not explode, it flatlines near **2.30**, and that number is not random. **ln(10) ≈ 2.30** is the cross-entropy of a model that spreads its probability uniformly across our ten classes. With last session's tool, perplexity = e^loss = e^2.30 ≈ 10: a one-in-ten guess, measured live. The oversized steps bulldozed the pre-trained weights, and the only strategy left to the wreckage is to hedge equally over every category. The failure is structural: fine-tuning works *because* it takes timid steps near already-good weights; take giant ones and you are training a randomly-initialised model on 748 examples, which cannot learn ten classes at all.

Now the real fine-tune at the safe learning rate. `compute_metrics` reports macro-F1 (our leaderboard metric) alongside accuracy so you can watch the gap between them on an imbalanced set.

In [ ]:
NUM_EPOCHS = 4   # try: 2  (how much of the final macro-F1 arrives in the first couple of epochs?)

import numpy as np
import evaluate

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

# Report both: accuracy hides the rare classes, macro-F1 (our leaderboard metric) does not.
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_metric.compute(predictions=preds, references=labels)["accuracy"],
        "macro_f1": f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"],
    }

ft_model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint, num_labels=len(CATEGORIES), id2label=id2label, label2id=label2id
)

ft_args = TrainingArguments(
    output_dir="./triage_full",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=0.01,
    eval_strategy="epoch",
    logging_steps=50,
    report_to="none",
    fp16=True,
    seed=SEED,
)

ft_trainer = Trainer(
    model=ft_model,
    args=ft_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

Training blocks the kernel for a few minutes, so the room works in parallel. The next cell prints the jury tickets again with no labels; lock in your routing now, because the reveal comes *after* training finishes. Printing has to happen before `train()` for exactly that reason.

In [ ]:
print("Route each ticket to ONE category and keep your answer:\n")
for i, text in enumerate(jury_df["text"], 1):
    print(f"{i:>2}. {text}")

In [ ]:
import time

t0 = time.time()
ft_trainer.train()
ft_train_seconds = time.time() - t0
print(f"\nTraining took {ft_train_seconds / 60:.1f} minutes.")

The reveal: the same tickets, now with the gold label and the fine-tuned model's verdict side by side. Count your score against the model's. The disagreements cluster on the multi-category probes, the ceiling step 4 warned about.

In [ ]:
import torch

def hf_predict(mdl, texts, batch_size=32):
    mdl.eval()
    device = next(mdl.parameters()).device
    preds = []
    for i in range(0, len(texts), batch_size):
        enc = tokenizer(list(texts[i:i + batch_size]), truncation=True,
                        padding=True, return_tensors="pt").to(device)
        with torch.no_grad():
            logits = mdl(**enc).logits
        preds.extend(logits.argmax(dim=-1).cpu().tolist())
    return [id2label[p] for p in preds]

jury_ft_preds = hf_predict(ft_model, jury_df["text"].tolist())

print(f"{'#':>2}  {'gold':<16} {'RoBERTa':<16} ticket")
print("-" * 90)
for i, (row, pred) in enumerate(zip(jury_df.itertuples(), jury_ft_preds), 1):
    flag = "ok  " if pred == row.category else "MISS"
    print(f"{i:>2}. [{flag}] {row.category:<14} {pred:<16} {row.text[:60]}")

In [ ]:
arena_texts = arena_df["text"].tolist()

t0 = time.time()
ft_preds = hf_predict(ft_model, arena_texts)
ft_ms = (time.time() - t0) / len(arena_texts) * 1000

ft_f1 = f1_score(arena_df["category"], ft_preds, average="macro")
print(f"Fine-tuned RoBERTa | arena macro-F1 = {ft_f1:.4f}")
post_score(f"RoBERTa full fine-tune ({NUM_EPOCHS} ep)", ft_f1, ft_train_seconds, ft_ms)

## Step 6: LoRA, how many of those 125M parameters did we need?

The full fine-tune rewrote every one of RoBERTa's ~125 million weights to learn what is, in the end, a ten-way routing decision. **LoRA** (Low-Rank Adaptation) bets that the *change* the task needs is low-rank: it freezes the pre-trained weights and learns small adapter matrices next to the attention projections. Watch the trainable-parameter count the next cell prints. That is the whole pitch.

In [ ]:
LORA_R = 16       # try: 8, 4  (watch macro-F1 fall as the rank shrinks)
LORA_EPOCHS = 8   # the base is frozen, so the adapters and the fresh head need several passes

from peft import LoraConfig, get_peft_model, TaskType

lora_base = AutoModelForSequenceClassification.from_pretrained(
    checkpoint, num_labels=len(CATEGORIES), id2label=id2label, label2id=label2id
)

# Freeze the base; learn only small rank-r adapters on the attention query/value projections.
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=LORA_R,
    lora_alpha=2 * LORA_R,
    lora_dropout=0.1,
    target_modules=["query", "value"],
    modules_to_save=["classifier"],   # also train the fresh head; LoRA alone leaves it random
)
lora_model = get_peft_model(lora_base, lora_config)
lora_model.print_trainable_parameters()

Under 1% of the model is trainable; the other 99%+ stays frozen exactly as pre-trained. Because the base is frozen, only the rank-16 adapters and the fresh classification head learn. They need a bit more room than the full fine-tune did: a slightly larger learning rate and several epochs (LORA_EPOCHS) to converge. Shrink LORA_R or LORA_EPOCHS and you will watch the score fall, which is the experiment those knobs invite.

In [ ]:
lora_args = TrainingArguments(
    output_dir="./triage_lora",
    learning_rate=1e-4,            # larger steps than the full fine-tune, but not so large the head collapses
    per_device_train_batch_size=16,
    num_train_epochs=LORA_EPOCHS,
    weight_decay=0.01,
    eval_strategy="epoch",
    logging_steps=50,
    report_to="none",
    fp16=True,
    seed=SEED,
)

lora_trainer = Trainer(
    model=lora_model,
    args=lora_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

t0 = time.time()
lora_trainer.train()
lora_train_seconds = time.time() - t0
print(f"\nLoRA training took {lora_train_seconds / 60:.1f} minutes.")

This adapter is the running-system artifact. We save it into the repo's artifact store (the megabyte-sized adapter plus the label map) so session 08's agent and session 09's capstone load *exactly this* tool. Then we compare its size against the full checkpoint.

In [ ]:
ADAPTER_DIR = "NLP-MBDS-EN/system/artifacts/06_finetuning/v1/triage_adapter"
lora_model.save_pretrained(ADAPTER_DIR)

save_artifact("06_finetuning", "label_map", {"id2label": id2label, "label2id": label2id})

print("Adapter on disk (the shipped triage tool):")
!ls -lh $ADAPTER_DIR
print()
print("Full fine-tuned checkpoint, for comparison:")
!ls -lh ./triage_full/checkpoint-*/model.safetensors

A few megabytes of adapter against roughly half a gigabyte of full checkpoint. That gap changes how models ship, and the demo below shows the mechanism. A fresh `roberta-base` with a brand-new, untrained classification head predicts essentially at random on one arena ticket, then we hot-swap the adapter onto it and ask again.

In [ ]:
from peft import PeftModel

DEMO_TEXT = arena_df["text"].iloc[0]
DEMO_GOLD = arena_df["category"].iloc[0]

def top_prediction(mdl, text):
    mdl.eval()
    device = next(mdl.parameters()).device
    enc = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        probs = torch.softmax(mdl(**enc).logits, dim=-1)[0]
    idx = int(probs.argmax())
    return id2label[idx], float(probs[idx])

print(f"Ticket: {DEMO_TEXT}")
print(f"Gold category: {DEMO_GOLD}\n")

fresh_base = AutoModelForSequenceClassification.from_pretrained(
    checkpoint, num_labels=len(CATEGORIES), id2label=id2label, label2id=label2id
)
label, conf = top_prediction(fresh_base, DEMO_TEXT)
print(f"fresh roberta-base head      → {label} ({conf:.2f})   [near-random: 1/{len(CATEGORIES)} ≈ {1/len(CATEGORIES):.2f}]")

swapped = PeftModel.from_pretrained(fresh_base, ADAPTER_DIR)
label, conf = top_prediction(swapped, DEMO_TEXT)
print(f"after attaching the adapter  → {label} ({conf:.2f})")

In [ ]:
lora_preds = hf_predict(lora_model, arena_texts)

t0 = time.time()
_ = hf_predict(lora_model, arena_texts)
lora_ms = (time.time() - t0) / len(arena_texts) * 1000

lora_f1 = f1_score(arena_df["category"], lora_preds, average="macro")
print(f"LoRA RoBERTa | arena macro-F1 = {lora_f1:.4f}")
post_score(f"LoRA RoBERTa (r={LORA_R}, {LORA_EPOCHS} ep)", lora_f1, lora_train_seconds, lora_ms)

## Takeaway

LoRA trains under 1% of the parameters and packs the whole learned task into a few megabytes. On the leaderboard it lands close to the full fine-tune; the gap is usually a point or two, and it widens if you lower LORA_R or LORA_EPOCHS. The deployment story matters more than the benchmark row: one frozen base model in production, and each team ships its task as a megabyte-sized adapter that hot-swaps onto it, with no half-gigabyte artifact per use case. And **this** adapter is not a demo. It now lives in the running system as Helcon's triage tool, the exact classifier session 08's agent will call to route a live ticket.

## Step 7: Zero-shot classification (no labels at all)

Every approach so far consumed our 748 labelled tickets. **Zero-shot classification** asks for none. A model pre-trained on Natural Language Inference decides whether a ticket *entails* a hypothesis built from each candidate label, so defining the labels is the entire "training". We feed it the taxonomy descriptions as richer hypotheses than the bare category names. Will a model that has never seen a single Helcon ticket get close to the supervised baseline?

In [ ]:
# The candidate labels are the taxonomy descriptions, richer than the bare category names.
# try: LABELS = CATEGORIES  (swap the descriptions for the raw names and watch the score move)
LABELS = [taxonomy["categories"][c]["description"] for c in CATEGORIES]
DESC_TO_CAT = {taxonomy["categories"][c]["description"]: c for c in CATEGORIES}   # map back to a category

from transformers import pipeline

zero_shot = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device_map="auto")

demo = arena_df["text"].iloc[0]
result = zero_shot(demo, LABELS)
print(f"Ticket: {demo}")
print(f"Top hypothesis: {result['labels'][0]}  →  {DESC_TO_CAT[result['labels'][0]]}")

Now the model meets the arena, the same validation tickets every other contender is judged on. A different sample, or tickets drawn from the training set, would make the leaderboard a lie even though zero-shot never trains: the comparison is only meaningful on identical data. There is no train time to report; we measure inference cost instead, and an NLI model runs once per candidate label per ticket, so it is not cheap.

In [ ]:
from tqdm.auto import tqdm

t0 = time.time()
nli_preds = []
for text in tqdm(arena_texts, desc="Zero-shot NLI on the arena"):
    top = zero_shot(text, LABELS)["labels"][0]
    nli_preds.append(DESC_TO_CAT[top])
nli_ms = (time.time() - t0) / len(arena_texts) * 1000

nli_f1 = f1_score(arena_df["category"], nli_preds, average="macro")
print(f"Zero-shot NLI | arena macro-F1 = {nli_f1:.4f}")
post_score("Zero-shot NLI (BART-MNLI)", nli_f1, None, nli_ms)

## Takeaway

A model that received no task-specific training at all lands within reach of the supervised stack, the zero-shot promise: zero labelling cost, paid for in inference cost (check the ms/ticket column). Two things to keep: fine-tuning on in-domain labels still pays, and the candidate labels are a hyperparameter. Swap the rich descriptions for the bare `CATEGORIES` names and the score moves on phrasing alone. The other label-free route is prompting an instruction-tuned LLM, next.

## Step 8: Prompting an instruction-tuned LLM

The NLI pipeline is not the only label-free route. The second is to talk to an instruction-tuned LLM through its native interface, a prompt: describe the task in plain language, list the taxonomy, and ask for exactly one category. The prompt *is* the editable artifact of this section. Change it, re-run, watch the macro-F1.

In [ ]:
# The prompt is the artifact here. Edit it, re-run the two cells below, watch the score.
# try: drop the description after each category name and see what those descriptions were worth.
_cat_block = "\n".join(
    f"- {c}: {taxonomy['categories'][c]['description']}" for c in CATEGORIES
)

PROMPT_TEMPLATE = f"""You are Helcon's support-ticket triage system.

Classify the customer ticket below into EXACTLY ONE of these categories:

{_cat_block}

Rules:
- Reply with only the category name in capitals, nothing else.
- If a ticket spans two categories, pick the single most important one.

Ticket:
{{text}}

Category:"""

print(PROMPT_TEMPLATE.format(text="<ticket goes here>"))

In [ ]:
llm = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    dtype=torch.float16,
    device_map="auto",
)
# Generate by token budget only; clearing the model's default max_length stops the
# "both max_new_tokens and max_length are set" warning.
llm.model.generation_config.max_length = None

def classify_with_llm(text):
    messages = [{"role": "user", "content": PROMPT_TEMPLATE.format(text=text)}]
    out = llm(messages, max_new_tokens=8, do_sample=False, return_full_text=False)
    answer = out[0]["generated_text"].upper()
    # Parse the free-text answer back to a known category (longest match wins).
    hits = [c for c in CATEGORIES if c in answer]
    if hits:
        return max(hits, key=len)
    return "GENERAL"   # fallback when the model says something off-menu

t0 = time.time()
llm_preds = [classify_with_llm(t) for t in tqdm(arena_texts, desc="Prompted LLM on the arena")]
llm_ms = (time.time() - t0) / len(arena_texts) * 1000

llm_f1 = f1_score(arena_df["category"], llm_preds, average="macro")
print(f"Prompted LLM | arena macro-F1 = {llm_f1:.4f}")
post_score("Prompted LLM (Qwen2.5-1.5B)", llm_f1, None, llm_ms)

### The triage leaderboard

Every approach, judged on the same 76-ticket arena.

In [ ]:
show_leaderboard()

## Step 9: When does fine-tuning actually earn its keep?

Look at the triage leaderboard: TF-IDF tied the fine-tuned transformers. That is the **honest result for a keyword task**. Routing a ticket to BILLING vs INTEGRATIONS is decided by words like *refund* or *webhook*, and a bag-of-words model finds those instantly. On this kind of task the 50-year-old cheap baseline is genuinely hard to beat, and reaching for a GPU buys you almost nothing.

But not every text task is keyword-separable. Step outside Helcon's triage for a moment to a task where the meaning lives in **context**, not vocabulary: deciding whether a message reports a *real* emergency. The word *disaster* is literal in "a wildfire is spreading near the city" and metaphorical in "this exam was a disaster": same word, opposite meaning. Same fine-tuning recipe as before; watch the outcome flip.

In [ ]:
import os
import pandas as pd

# The disaster-tweets dataset ships in the repo (real Kaggle data, binary labels).
_cands = ["NLP-MBDS-EN/06_transfer_learning_and_fine_tuning/nlp_disaster/data/train.csv",
          "nlp_disaster/data/train.csv",
          "06_transfer_learning_and_fine_tuning/nlp_disaster/data/train.csv"]
disaster_path = next((p for p in _cands if os.path.exists(p)), _cands[0])
disaster = pd.read_csv(disaster_path).dropna(subset=["text", "target"])
disaster["target"] = disaster["target"].astype(int)

print(f"{len(disaster)} tweets  |  label 1 = real disaster, 0 = not")
print("\nliteral  (1):", disaster[disaster.target == 1]["text"].iloc[0])
print("metaphor (0):", disaster[disaster.target == 0]["text"].iloc[3])

In [ ]:
# Same TF-IDF + LogisticRegression recipe, on a separate little leaderboard.
DISASTER_SAMPLE = 4000   # subsample to keep training quick; try: None for the full ~7600

d = disaster if DISASTER_SAMPLE is None else disaster.sample(DISASTER_SAMPLE, random_state=SEED)
d_tr, d_va = train_test_split(d, test_size=0.2, random_state=SEED, stratify=d["target"])

HARD = {}
def post_hard(name, macro_f1, train_seconds=None):
    HARD[name] = {"macro_f1": macro_f1, "train_seconds": train_seconds}
    print(f"posted: {name} (macro-F1 = {macro_f1:.3f})")

clf_d = Pipeline([
    ("tfidf", TfidfVectorizer(preprocessor=preprocess, ngram_range=(1, 2), min_df=2)),
    ("lr", LogisticRegression(max_iter=1000, C=10.0)),
])
t0 = time.time()
clf_d.fit(d_tr["text"], d_tr["target"])
d_tfidf_seconds = time.time() - t0
d_tfidf_f1 = f1_score(d_va["target"], clf_d.predict(d_va["text"]), average="macro")
post_hard("TF-IDF + LogReg", d_tfidf_f1, d_tfidf_seconds)

On triage, TF-IDF scored near the top. Here it lands much lower: bag-of-words can see the word *disaster*, but not whether it means an emergency or a bad day. Before running the next cell: will fine-tuning RoBERTa beat it this time?

In [ ]:
from datasets import Dataset, DatasetDict
import torch

DISASTER_EPOCHS = 2   # try: 3

d_id2label = {0: "not disaster", 1: "real disaster"}
d_label2id = {v: k for k, v in d_id2label.items()}

def _to_hf(df):
    return Dataset.from_pandas(df[["text"]].assign(label=df["target"].values), preserve_index=False)

d_ds = DatasetDict({"train": _to_hf(d_tr), "validation": _to_hf(d_va)})
d_tok = d_ds.map(lambda b: tokenizer(b["text"], truncation=True), batched=True)

d_model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint, num_labels=2, id2label=d_id2label, label2id=d_label2id
)
d_args = TrainingArguments(
    output_dir="./disaster_ft", learning_rate=2e-5, per_device_train_batch_size=16,
    num_train_epochs=DISASTER_EPOCHS, weight_decay=0.01, eval_strategy="epoch",
    logging_steps=100, report_to="none", fp16=True, seed=SEED,
)
d_trainer = Trainer(
    model=d_model, args=d_args, train_dataset=d_tok["train"], eval_dataset=d_tok["validation"],
    data_collator=data_collator, compute_metrics=compute_metrics,
)
t0 = time.time()
d_trainer.train()
d_ft_seconds = time.time() - t0

def predict_ids(model, texts, batch_size=32):
    model.eval()
    device = next(model.parameters()).device
    preds = []
    for i in range(0, len(texts), batch_size):
        enc = tokenizer(texts[i:i + batch_size], truncation=True, padding=True, return_tensors="pt").to(device)
        with torch.no_grad():
            preds += model(**enc).logits.argmax(-1).tolist()
    return preds

d_ft_f1 = f1_score(d_va["target"], predict_ids(d_model, d_va["text"].tolist()), average="macro")
post_hard(f"RoBERTa fine-tune ({DISASTER_EPOCHS} ep)", d_ft_f1, d_ft_seconds)

In [ ]:
print(f"{'approach':<32}{'macro-F1':>10}{'train (s)':>11}")
print("-" * 53)
for name, r in sorted(HARD.items(), key=lambda kv: kv[1]["macro_f1"], reverse=True):
    ts = "-" if r["train_seconds"] is None else f"{r['train_seconds']:.0f}"
    print(f"{name:<32}{r['macro_f1']:>10.3f}{ts:>11}")

## Takeaway

The result flipped. On triage, TF-IDF tied the transformers; here the fine-tuned model clearly **beats** it, because telling a real emergency from a figure of speech needs the surrounding context, exactly what bag-of-words throws away and what a pretrained transformer keeps.

So the two halves of the session together give the real lesson, and it is not "transformers always win":

- **Match the model to the task.** When the signal is in the *vocabulary* (triage), the cheap, instant, GPU-free baseline is the right call. When the signal is in the *context* (literal vs figurative, sentiment, sarcasm, intent under ambiguity), fine-tuning earns its cost.
- The professional skill is knowing which kind of task you have **before** you spend the GPU budget, and a TF-IDF baseline is the cheapest way to find out.